### Load Dataset ### 

In [2]:
import pandas as pd 
import numpy as np 

df = pd.read_csv('../data/processed/master_dataset_processed.csv')
df.head()

,label,message,source,scam_type,message_length,word_count,clean_message,processed_message
0,legit,"Go until jurong point, crazy.. Available only ...",sms,none,111,20,go until jurong point crazy available only in ...,jurong point crazi avail bugi great world buff...
1,legit,Ok lar... Joking wif u oni...,sms,none,29,6,ok lar joking wif u oni,lar joke wif oni
2,scam,Free entry in 2 a wkly comp to win FA Cup fina...,sms,unknown,155,28,free entry in a wkly comp to win fa cup final ...,free entri wkli comp win cup final tkt may tex...
3,legit,U dun say so early hor... U c already then say...,sms,none,49,11,u dun say so early hor u c already then say,dun say earli hor alreadi say
4,legit,"Nah I don't think he goes to usf, he lives aro...",sms,none,61,13,nah i don t think he goes to usf he lives arou...,nah think goe usf live around though


In [3]:
df['processed_message'].isnull().sum()

np.int64(187)

In [4]:
df['processed_message'] = df['processed_message'].fillna('')

### Fit TF-IDF on your processed text ###

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer 

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=5, max_df=0.8)

tfidf_matrix = tfidf.fit_transform(df['processed_message'])
tfidf_matrix.shape

(49019, 5000)

### Manual/Engineered Features ###


### Feature 1: Urgency language ###

In [6]:
urgency_words = ['urgent', 'immediately', 'act now', 'hurry', 'asap', 'expire', 'expires', 'expiring', 'limited time', 'last chance', 'final notice', 'deadline', 'today only']

def has_urgency(text):
    text = str(text).lower()
    return int(any(word in text for word in urgency_words))

df['has_urgency'] = df['message'].apply(has_urgency)

### Feature 2: Authority claims ###

In [7]:
authority_words = ['police', 'cbi', 'income tax', 'court', 'legal action', 'government', 'customs', 'bank official', 'rbi', 'arrest warrant', 'officer', 'department', 'federal', 'irs']

def has_authority_claim(text):
    text = str(text).lower()
    return int(any(word in text for word in authority_words))

df['has_authority_claim'] = df['message'].apply(has_authority_claim)


### Feature 3: Financial Requests ###

In [8]:
financial_words = ['transfer', 'wire', 'payment', 'pay now', 'bank account', 'account number', 'deposit', 'refund', 'fee', 'processing fee', 'claim your', 'send money', 'western union']

def has_financial_request(text):
    text = str(text).lower()
    return int(any(word in text for word  in financial_words))

df['has_financial_request'] = df['message'].apply(has_financial_request)

### Feature 4: OTP Password Credential Requests ###

In [9]:
credential_words = ['enter your otp', 'share your otp', 'verify your account', 'confirm your identity', 'enter your pin', 'enter your password', 'cvv', 'card number', 'security code', 'social security number', 'ssn']

def has_credential_request(text):
    text = str(text).lower()
    return int(any(word in text for word in credential_words))

df['has_credential_request'] = df['message'].apply(has_credential_request)

### Feature 6: Contains URL, Phone number, Email ###

In [10]:
import re 

df['has_url'] = df['message'].apply(lambda x: int(bool(re.search(r'http\S+', str(x)))))
df['has_phone_number'] = df['message'].apply(lambda x: int(bool(re.search(r'\b\d{10,}\b|\+\d{1,3}}[\s-]?\d{6,}', str(x)))))
df['has_email'] = df['message'].apply(lambda x: int(bool(re.search(r'\S+@\S+\.\S+', str(x)))))

### Feature 7: Excessive punctuation ###

In [11]:
df['exclamation_count'] = df['message'].apply(lambda x: str(x).count('!'))
df['caps_word_count'] = df['message'].apply(lambda x: sum(1 for w in str(x).split() if w.isupper() and len(w) > 2))

In [12]:
feature_cols = ['has_urgency', 'has_authority_claim', 'has_financial_request', 'has_credential_request', 'has_url', 'has_phone_number', 'has_email']

df.groupby('label')[feature_cols].mean()

,has_urgency,has_authority_claim,has_financial_request,has_credential_request,has_url,has_phone_number,has_email
label,,,,,,,
legit,0.093781,0.254353,0.186990,0.004353,0.097236,0.012325,0.075879
scam,0.229063,0.331403,0.416456,0.014189,0.167985,0.068091,0.142787


In [13]:
df.to_csv('../data/processed/master_dataset_features.csv', index=False)

### Combine TF-IDF + manual features ###

In [15]:
from scipy.sparse import hstack, csr_matrix

manual_features = df[['message_length', 'word_count', 'has_urgency', 'has_authority_claim',
                       'has_financial_request', 'has_credential_request', 'has_url',
                       'has_phone_number', 'has_email', 'exclamation_count', 'caps_word_count']].values

manual_features_sparse = csr_matrix(manual_features)

final_features = hstack([tfidf_matrix, manual_features_sparse])
final_features.shape

(49019, 5011)